# SailGP Data Analyst Challenge

The aim is to test you python abilities. The challenge is to analyze the data provided and answer the questions below. You can use any library you want to help you with the analysis. The data is from the SailGP event in Auckland 2025. The data is in the 'DATA' folder.

There are various sources available.

The Boat Logs are in the 'Boat_Logs' folder. The data is in csv format and the columns are described in the 'Boat_Logs/Boat_Logs_Columns.csv' file.
The 'Course_Marks_2025-01-19.csv' file contains the mark positions and wind reading on the course for the whole day.

The Race_XML folder contains the xml files for each race that contains information on where the boundaries of the course are, the theoretical position of the marks and the target racecourse axis.

The 2025-01-19_man_summary.csv file contains the metrics from the manoeuvre summary for the day.
The 2025-01-19_straight_lines.csv file contains the metrics from the straight line summary for the day.

Both are derived from the boat logs.

The 2502 m8_APW_HSB2_HSRW.kph.csv file contains the polar data for the boats in that config.

## Requierements
- Chose at least 3 questions from the list below to answer.
- Python 3.8 or higher
- Notebook should be able to run without any errors from start to finish.
- Specify the libraries (imports) used in the notebook.
- Any comments to make the notebook self-explanatory and easy to follow would be appreciated.
- If you can't get to the end of a question, we would appreciate the code you have written so far and explain what you were trying to do.

## Further information:
- We usually use bokeh for visualizations. So any showcase of bokeh would be appreciated.
-

## Submitting the results.
It would be great if you could provide a jupyter notebook with the code and the results of the analysis. You can submit the results by sharing a link to a git repository.


### Imports and re-used functions
Free section to initialize the notebook with the necessary imports and functions that will be used in the notebook.



In [344]:
#Importing all recommended Libraries

import numpy as np
import pandas as pd
import bokeh
import matplotlib
import scipy
from bs4 import BeautifulSoup
import math
from typing import List, Tuple

from bokeh.plotting import figure, show
from bokeh.io import output_file
from bokeh.models import Range1d, DatetimeTickFormatter
from bokeh.palettes import Category20


import unittest
import math

## Question 1: Write a Python function that can take a compass direction (ie. TWD or Heading) and calculate an accurate mean value across a downsampled frequency. Eg. If TWD is at 1Hz, give me a 10s average.

In [345]:
'''
Calculating mean value for TWD for GBR's boat log

Thoughts on question:
    Direction is in degrees so might have issue with 359degrees 
    averaging with 1 degrees to make 180 mean which is totally wrong.
'''


def calculate_average_bearing(bearings: List[float]) -> float:
    '''
    param: bearings: list of angles to be averaged. In degrees as stored as floats
    return: a single float that is the average, given back as a positive between 0 and 360, in degrees rounded to two decimal places.
    
    This function will find the average bearing from a list of bearings. It works by turning each bearing into a unit vector,
    adding up all the unit vectors and then taking the bearing of the final vector.
    '''
    #Finding the x and y length of each vector from the bearings
    x_coords = [math.sin(math.radians(bearing)) for bearing in bearings]
    y_coords = [math.cos(math.radians(bearing)) for bearing in bearings]
    
    #adding the x and y respectivly
    x_tot = sum(x_coords)
    y_tot = sum(y_coords)
    
    #finding the final angle of the total vector using tan^(-1)
    angle_in_degrees = round(math.degrees(math.atan2(x_tot, y_tot)),2)

    return (angle_in_degrees + 360) % 360


def resample_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    '''
    param: df: a dataframe to be resampled
    return: a dataframe that is resampled to 1 second.
    '''
    df['DATETIME'] = pd.to_datetime(df['DATETIME'])
    df = df.sort_values(by='DATETIME')
    df_resampled = df.set_index('DATETIME').resample('1s').first()
    return df_resampled


#Loading in the data to a dataframe
df = pd.read_csv("Data/Boat_Logs/data_GBR.csv",usecols=["TWD_SGP_deg","DATETIME"])

#resampling to 1 second intervals as there are gaps.
df = resample_timestamps(df)

#calculating an average wind speed for ten second sections
df['rolling_mean_wind'] = df['TWD_SGP_deg'].rolling(window='10s').apply(calculate_average_bearing)
pd.set_option('display.max_rows', 100)
print(df.tail(20))


                     TWD_SGP_deg  rolling_mean_wind
DATETIME                                           
2025-01-19 04:22:56        66.46              65.97
2025-01-19 04:22:57        66.49              66.14
2025-01-19 04:22:58        66.53              66.27
2025-01-19 04:22:59        66.63              66.37
2025-01-19 04:23:00        66.51              66.43
2025-01-19 04:23:01        66.41              66.45
2025-01-19 04:23:02        66.33              66.45
2025-01-19 04:23:03        66.15              66.43
2025-01-19 04:23:04        65.90              66.38
2025-01-19 04:23:05        65.60              66.30
2025-01-19 04:23:06        65.28              66.18
2025-01-19 04:23:07        64.98              66.03
2025-01-19 04:23:08        64.68              65.85
2025-01-19 04:23:09        64.45              65.63
2025-01-19 04:23:10        64.22              65.40
2025-01-19 04:23:11        63.80              65.14
2025-01-19 04:23:12        63.42              64.85
2025-01-19 0

In [346]:
#SCRAP working that has been cleaned up above.


#Calculating mean value for TWD for GBR's boat log

#Thoughts on question:
# * Direction is in degrees so might have issue with 359degrees 
#   averaging with 1 degrees to make 180 mean which is totally wrong.


def calculate_mean_bearing(bearings: List[float]) -> float:
    '''
    Param: bearings: List of floats to take the mean average of, these must be bearings in degrees
    Return: float of average bearing in degrees.
    
    Function works by adding unit vectors together and then taking the bearing of the total vector.
    '''
    x_total = 0
    y_total = 0
    for angle in bearings:
        angle = math.radians(angle)
        x_total += math.sin(angle)
        y_total += math.cos(angle)
    mean_angle = math.degrees(math.atan2(x_total, y_total))
    if mean_angle < 0:
        mean_angle += 360
    return mean_angle

def improved(bearings: List[float]) -> float:
    x_coords = [math.sin(math.radians(bearing)) for bearing in bearings]
    y_coords = [math.cos(math.radians(bearing)) for bearing in bearings]
    
    x_tot = sum(x_coords)
    y_tot = sum(y_coords)
    
    angle_in_degrees = round(math.degrees(math.atan2(x_tot, y_tot)),2)

    return (angle_in_degrees + 360) % 360

print(calculate_mean_bearing([150, 75, 38, 10]))
print(improved([10, 20, 30]))

'''
def drop_consecutive_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    df['DATETIME'] = pd.to_datetime(df['DATETIME'])
    df["time_diff"] = df['DATETIME'].diff().dt.total_seconds().shift(-1)
    print(df["time_diff"].value_counts())
    filtered_df = df[df["time_diff"] < 2]
    return filtered_df


time between - number of entries
1.0      2737
370.0       1
557.0       1
32.0        1
577.0       1
732.0       1

essentially gaps in measurements
'''


def resample_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    df['DATETIME'] = pd.to_datetime(df['DATETIME'])
    df = df.sort_values(by='DATETIME')
    df_resampled = df.set_index('DATETIME').resample('1s').first()
    return df_resampled


df = pd.read_csv("Data/Boat_Logs/data_GBR.csv",usecols=["TWD_SGP_deg","DATETIME"])

df = resample_timestamps(df)

df['rolling_bearing'] = df['TWD_SGP_deg'].rolling(window='10s').apply(lambda x: improved(x), raw=False)
pd.set_option('display.max_rows', 100)
print(df.tail(20))

62.66794483096409
20.0
                     TWD_SGP_deg  rolling_bearing
DATETIME                                         
2025-01-19 02:59:50        59.35            59.35
2025-01-19 02:59:51          NaN              NaN
2025-01-19 02:59:52          NaN              NaN
2025-01-19 02:59:53          NaN              NaN
2025-01-19 02:59:54          NaN              NaN
2025-01-19 02:59:55          NaN              NaN
2025-01-19 02:59:56          NaN              NaN
2025-01-19 02:59:57          NaN              NaN
2025-01-19 02:59:58          NaN              NaN
2025-01-19 02:59:59          NaN              NaN
2025-01-19 03:00:00          NaN              NaN
2025-01-19 03:00:01          NaN              NaN
2025-01-19 03:00:02          NaN              NaN
2025-01-19 03:00:03          NaN              NaN
2025-01-19 03:00:04          NaN              NaN
2025-01-19 03:00:05          NaN              NaN
2025-01-19 03:00:06          NaN              NaN
2025-01-19 03:00:07        

## Question 2: Given a course XML and a timeseries of boat Lat/Lon values, calculate a VMC column for the same timeseries.


In [347]:
'''
assumed the earth is flat as the earth curvature will have negligable effect on this size of race.
would affect it for a race around the world like the clipper race.


using speed over ground rather than speed as im using locations so more accurate, gets rid of tides.


With the gates on the course it will do a target to the midpoint of the two markers.

underlying formula:
boat speed * cos(angle difference between course and target)
'''

def calculate_midpoint(gate: [(float, float),(float, float)]) -> (float,float):
    '''
    param: gate: takes two coordinates as a list of float tuples.
    return: a tuple of floats of x and y position that is the midpoint
    This function will just find the midpoint of two coordinates by adding half the difference to the first coordinate.
    '''
    return (gate[0][0]+((gate[1][0]-gate[0][0])/2),gate[0][1]+((gate[1][1]-gate[0][1])/2))


def get_marker_coordinates(compound_marks):
    '''
    param: compound_marks: takes in xml
    return: returns a list of coordinates
    This function just goes through the XML and finds the marks coordinates and puts them into a list.
    Looks for keywords "Mark", "TargetLat" and "TargetLng"
    '''
    coordinates = []
    
    for compound_mark in compound_marks:
        if isinstance(compound_mark, str):
            continue
        
        compound_coordinates = []
        
        marks = compound_mark.find_all('Mark')
        
        for mark in marks:
            lat = float(mark['TargetLat'])
            lng = float(mark['TargetLng'])
            compound_coordinates.append((lat, lng))
        if len(compound_coordinates) == 2:
            compound_coordinates = calculate_midpoint(compound_coordinates)
        else:
            compound_coordinates = compound_coordinates[0]
        coordinates.append(compound_coordinates)
    
    return coordinates


def calculate_line_bearing(a: [float,float], b: [float,float]) -> float:
    '''
    param: a: a list of two floats that is the coordinates for point a
    param: b: a list of two floats that is the coordinates for point b
    return: returns an angle in degrees between 0 and 360 that is the bearing of point b from point a
    '''
    x = b[0] - a[0]
    y = b[1] - a[1]

    bearing = math.degrees(math.atan2(math.radians(y), math.radians(x)))
    bearing = (bearing + 360) % 360

    return bearing



'''
old function
Calculates vmc of line at gate. but decided i actually just want the midpoint.


def calculate_VMC_with_gate(boat_heading: float, boat_speed: float, gate_bearing: float) -> float:
    angle_diff = abs(heading - gate_bearing)
    if angle_diff > 180:
        angle_diff = 360 - angle_diff

    angle_diff_rad = math.radians(angle_diff)

    vmg = speed * math.cos(angle_diff_rad)
    return vmg


'''



def calculate_vmc(boat_speed: float, boat_heading: float, target_bearing: float) -> float:
    '''
    param: boat_speed: the boats speed as a float.
    param: boat_heading: the boats heading in degrees as a float.
    param: target_bearing: the bearing of the target in degrees as a float.
    return: returns the vmc of the boat towards the target as a float in same measurement of speed that it is given.
    this function will calculate the vmc of the boat towards a target.
    
    
    side note: in all honesty this could be inside the get_vmc function however it shows the underlying algorithm in simplest
    terms so thats why i have it seperate.
    '''
    return boat_speed * math.cos(math.radians(boat_heading - target_bearing))



def get_vmc(marker_coordinates: [], boat_lattitude: float, boat_longitude: float, boat_heading: float, boat_speed: float, boat_leg: int) -> float:
    '''
    param: marker_coordinates: coordinates of all the markers - gates accepted as a single marker - midpoint used.
    param: boat_lattitude: boats lattitude position as float
    param: boat_longitude: boats longitude position as float
    param: boat_heading: boats current heading as a float
    param: boat_speed: boats speed as a float
    param: boat_leg: boats current leg in the race, to work out which target is next.
    return: float that is the current vmc of a boat towards its next target.
    '''
    target_num = boat_leg
    target_bearing = calculate_line_bearing([boat_lattitude, boat_longitude], marker_coordinates[int(target_num)])
    return calculate_vmc(boat_speed, boat_heading, target_bearing)



#reading in the files below

with open('Data/Race_XMLs/25011905_03-13-55.xml', 'r') as file:
    race = BeautifulSoup(file, 'xml')  # 'xml' is the parser mode
course = race.find('Course')

df = pd.read_csv("Data/Boat_Logs/data_GBR.csv",usecols=["LATITUDE_GPS_unk","LONGITUDE_GPS_unk","HEADING_deg","GPS_SOG_km_h_1","TRK_LEG_NUM_unk"])

#turning the gates into single target coordinates
marker_coordinates = get_marker_coordinates(course)
print(marker_coordinates)

#getting the vmc for each point.
df['VMC'] = df.apply(lambda row: get_vmc(
    marker_coordinates,
    row['LATITUDE_GPS_unk'],
    row['LONGITUDE_GPS_unk'],
    row['HEADING_deg'],
    row['GPS_SOG_km_h_1'],
    row['TRK_LEG_NUM_unk']
), axis=1)

#Showing the new column has been added.
print(df)


[(-36.834987, 174.7686595), (-36.829713, 174.76527), (-36.8338455, 174.7548255), (-36.829794, 174.767608), (-36.8338455, 174.7548255), (-36.829794, 174.767608), (-36.834715, 174.755873), (-36.8358705, 174.75856149999998)]
      LATITUDE_GPS_unk  LONGITUDE_GPS_unk  GPS_SOG_km_h_1  HEADING_deg  \
0           -36.829452         174.757544           54.66       266.71   
1           -36.834131         174.758426           29.77       112.60   
2           -36.834163         174.758509           30.01       116.38   
3           -36.834200         174.758592           31.68       117.02   
4           -36.834238         174.758679           32.84       116.16   
...                ...                ...             ...          ...   
2738        -36.835545         174.758146           44.44        93.85   
2739        -36.835546         174.758283           42.80        92.98   
2740        -36.835547         174.758416           40.77        94.43   
2741        -36.835551         174.758

In [348]:
'''
just to show that I can unit test, with more time I would have unit tested everything:
the unit test does fail but only because the limitation on computers calculation. 
it gives a miniscule number instead of 0 that could be rounded to 0. but its fun to see.
'''



class TestCalculateVMC(unittest.TestCase):
    
    def test_zero_speed(self):
        # If the boat speed is 0, the VMC should also be 0 regardless of heading and target bearing
        self.assertEqual(calculate_vmc(0, 90, 180), 0)

    def test_same_heading_and_target_bearing(self):
        # If the boat heading and the target bearing are the same, the VMC should equal the boat speed
        self.assertEqual(calculate_vmc(10, 90, 90), 10)

    def test_opposite_heading_and_target_bearing(self):
        # If the boat heading is 180 degrees opposite the target bearing, the VMC should be negative
        self.assertEqual(calculate_vmc(10, 0, 180), -10)

    def test_heading_perpendicular_to_target_bearing(self):
        # If the boat heading is 90 degrees to the target bearing, the VMC should be 0 (no progress)
        self.assertEqual(calculate_vmc(10, 90, 0), 0)
        self.assertEqual(calculate_vmc(10, 90, 180), 0)

    def test_non_zero_vmc(self):
        # Check with random boat speed, heading, and target bearing values
        boat_speed = 10
        boat_heading = 45
        target_bearing = 0
        # The expected VMC would be: 10 * cos(45 - 0) = 10 * cos(45 degrees)
        expected_vmc = 10 * math.cos(math.radians(45))
        self.assertAlmostEqual(calculate_vmc(boat_speed, boat_heading, target_bearing), expected_vmc, places=7)

    def test_edge_case_360_degrees(self):
        # Test with boat heading and target bearing being 360 degrees (effectively equivalent to 0 degrees)
        boat_speed = 10
        boat_heading = 360
        target_bearing = 0
        # This should be the same as having a heading of 0, so expected VMC = boat_speed
        self.assertEqual(calculate_vmc(boat_speed, boat_heading, target_bearing), boat_speed)

    def test_large_numbers(self):
        # Test with large boat speed values
        boat_speed = 1000
        boat_heading = 30
        target_bearing = 60
        expected_vmc = 1000 * math.cos(math.radians(30 - 60))
        self.assertAlmostEqual(calculate_vmc(boat_speed, boat_heading, target_bearing), expected_vmc, places=7)



if __name__ == "__main__":
    # This allows the tests to run properly in various environments
    unittest.TextTestRunner().run(unittest.defaultTestLoader.loadTestsFromTestCase(TestCalculateVMC))


.F.....
FAIL: test_heading_perpendicular_to_target_bearing (__main__.TestCalculateVMC)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\andre\AppData\Local\Temp\ipykernel_23216\525087046.py", line 25, in test_heading_perpendicular_to_target_bearing
    self.assertEqual(calculate_vmc(10, 90, 0), 0)
AssertionError: 6.123233995736766e-16 != 0

----------------------------------------------------------------------
Ran 7 tests in 0.012s

FAILED (failures=1)


## Question 3: Verify and comment on the boats calibration. If possible propose a post-calibrated set of wind numbers and a potential calibration table.

In [343]:
'''
Analysing both wind speed and direction from the boats.
Taking the mean from all boats to be a reference and then judging each boat on the reference.
Getting RMSE for size of how far off it is and getting the mean difference to know which side of reference.
proposing a correction amount for each boats wind speed and direction.
'''

boat_files = [
    "Data/Boat_Logs/data_AUS.csv",
    "Data/Boat_Logs/data_GBR.csv",
    "Data/Boat_Logs/data_BRA.csv",
    "Data/Boat_Logs/data_CAN.csv",
    "Data/Boat_Logs/data_DEN.csv",
    "Data/Boat_Logs/data_ESP.csv",
    "Data/Boat_Logs/data_GER.csv",
    "Data/Boat_Logs/data_ITA.csv",
    "Data/Boat_Logs/data_NZL.csv",
    "Data/Boat_Logs/data_SUI.csv",
    "Data/Boat_Logs/data_USA.csv"
]

def load_boat_data(boat_files: list) -> pd.DataFrame:
    '''
    Loads and concatenates wind data for all boats into a single DataFrame.
    
    param: boat_files: A list of file paths to the boat data CSVs.
    return: A DataFrame containing the combined data for all boats with columns DATETIME, TWS_SGP_km_h_1, TWD_SGP_deg.
    '''
    all_boats_data = []
    for boat_file in boat_files:
        boat_data = pd.read_csv(boat_file, usecols=["DATETIME", "TWS_SGP_km_h_1", "TWD_SGP_deg"])
        boat_data['DATETIME'] = pd.to_datetime(boat_data['DATETIME'])
        all_boats_data.append(boat_data)
    return pd.concat(all_boats_data, ignore_index=True)



def calculate_average_reference(all_boats_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Calculates the average True Wind Speed (TWS) and True Wind Direction (TWD) across all boats
    to create the reference data.
    
    param: all_boats_df: DataFrame containing concatenated wind data for all boats.
    return: A DataFrame containing the average TWS and TWD at each timestamp.
    '''
    average_data = all_boats_df.groupby('DATETIME').agg({
        'TWS_SGP_km_h_1': 'mean',
        'TWD_SGP_deg': 'mean'
    }).reset_index()
    average_data.rename(columns={
        'TWS_SGP_km_h_1': 'TWS_ref_km_h',
        'TWD_SGP_deg': 'TWD_ref_deg'
    }, inplace=True)
    return average_data




def merge_and_calculate_diff(boat_data: pd.DataFrame, reference_data: pd.DataFrame) -> pd.DataFrame:
    '''
    Merges each boat's data with the reference (average) data and calculates the differences
    for TWS (True Wind Speed) and TWD (True Wind Direction).
    
    param: boat_data: A DataFrame containing a boat's wind data.
    param: reference_data: A DataFrame containing the average (reference) TWS and TWD data.
    return: A DataFrame containing the merged data with columns for TWS_diff and TWD_diff.
    '''
    merged_data = pd.merge(boat_data, reference_data, on="DATETIME", how="inner")
    merged_data['TWS_diff'] = merged_data['TWS_SGP_km_h_1'] - merged_data['TWS_ref_km_h']
    merged_data['TWD_diff'] = merged_data['TWD_SGP_deg'] - merged_data['TWD_ref_deg']
    return merged_data


def calculate_rmse_and_plot(merged_data: pd.DataFrame, boat_id: int, p: figure, colors: list) -> None:
    '''
    Calculates the Root Mean Square Error (RMSE) and mean differences for both TWS and TWD,
    and plots the differences for each boat on the same graph.
    
    param: merged_data: A DataFrame containing the merged boat and reference data with TWS_diff and TWD_diff.
    param: boat_id: An integer representing the boat index (used for labeling and colors).
    param: p: A Bokeh figure object for plotting.
    param: colors: A list of colors used to differentiate the boat plots.
    return: None
    '''
    mean_tws_diff = merged_data['TWS_diff'].mean()
    rmse_tws = np.sqrt((merged_data['TWS_diff']**2).mean())
    mean_twd_diff = merged_data['TWD_diff'].mean()
    rmse_twd = np.sqrt((merged_data['TWD_diff']**2).mean())
    
    print(f"Boat {boat_id} - TWS Mean Diff: {mean_tws_diff} km/h, RMSE: {rmse_tws} km/h")
    print(f"Boat {boat_id} - TWD Mean Diff: {mean_twd_diff} degrees, RMSE: {rmse_twd} degrees")
    
    correction_table = {
        'TWS_correction': mean_tws_diff,
        'TWD_correction': mean_twd_diff
    }
    print(f"Boat {boat_id} - Proposed Calibration Corrections: {correction_table}")
    
    p.line(merged_data['DATETIME'], merged_data['TWS_diff'], color=colors[boat_id], 
           legend_label=f"Boat {boat_id + 1}: TWS Diff", line_width=2)
    p.line(merged_data['DATETIME'], merged_data['TWD_diff'], color=colors[boat_id], 
           legend_label=f"Boat {boat_id + 1}: TWD Diff", line_width=2, y_range_name="twd_range")

    
    
    
def plot_data(boat_files: list, all_boats_df: pd.DataFrame, reference_data: pd.DataFrame) -> None:
    '''
    Plots the differences in True Wind Speed (TWS) and True Wind Direction (TWD) for each boat
    relative to the average reference data.
    
    param: boat_files: A list of file paths to the boat data CSVs.
    param: all_boats_df: DataFrame containing concatenated wind data for all boats.
    param: reference_data: DataFrame containing the average (reference) TWS and TWD data.
    return: None
    '''
    output_file("wind_data_comparison_all_boats.html")

    p = figure(x_axis_type='datetime', title="Wind Data Comparison: All Boats vs Reference",
               width=1000, height=600)

    p.extra_y_ranges = {"twd_range": Range1d(start=0, end=360)}
    p.add_layout(p.yaxis[0], 'left')

    colors = Category20[20]

    for i, boat_file in enumerate(boat_files):
        boat_data = pd.read_csv(boat_file, usecols=["DATETIME", "TWS_SGP_km_h_1", "TWD_SGP_deg"])
        boat_data['DATETIME'] = pd.to_datetime(boat_data['DATETIME'])

        merged_data = merge_and_calculate_diff(boat_data, reference_data)

        calculate_rmse_and_plot(merged_data, i, p, colors)

    p.xaxis.axis_label = 'Time'
    p.yaxis.axis_label = 'Difference (TWS and TWD)'
    p.yaxis[1].axis_label = 'TWD (degrees)'

    p.xaxis.formatter = DatetimeTickFormatter(hours="%H:%M", days="%d %b %H:%M")
    p.xaxis.major_label_orientation = 3.14 / 4

    p.legend.location = "top_left"
    p.legend.title = "Boats' Wind Data"

    show(p)


all_boats_df = load_boat_data(boat_files)
reference_data = calculate_average_reference(all_boats_df)
plot_data(boat_files, all_boats_df, reference_data)



Boat 0 - TWS Mean Diff: 0.2585035705193758 km/h, RMSE: 2.5487717706088793 km/h
Boat 0 - TWD Mean Diff: -0.2866530490725279 degrees, RMSE: 4.563723039683896 degrees
Boat 0 - Proposed Calibration Corrections: {'TWS_correction': np.float64(0.2585035705193758), 'TWD_correction': np.float64(-0.2866530490725279)}
Boat 1 - TWS Mean Diff: 1.1491922344374952 km/h, RMSE: 2.9140933476904443 km/h
Boat 1 - TWD Mean Diff: 1.1649002141881286 degrees, RMSE: 4.4083275412147485 degrees
Boat 1 - Proposed Calibration Corrections: {'TWS_correction': np.float64(1.1491922344374952), 'TWD_correction': np.float64(1.1649002141881286)}
Boat 2 - TWS Mean Diff: 0.0641453314639291 km/h, RMSE: 2.8772111664881246 km/h
Boat 2 - TWD Mean Diff: -0.7392429132520595 degrees, RMSE: 4.127162107774096 degrees
Boat 2 - Proposed Calibration Corrections: {'TWS_correction': np.float64(0.0641453314639291), 'TWD_correction': np.float64(-0.7392429132520595)}
Boat 3 - TWS Mean Diff: -2.5794538699092837 km/h, RMSE: 5.233090483337673 

## Question 4: Given a timeseries of Lat/Lon positions and a course XML, in a Python notebook, calculate a Distance to Leader metric for each boat.

## Question 5: Given a course XML, along with a wind speed and direction and a polar, calculate the minimum number of tacks or gybes for each leg of the course and each gate mark on the leg.

## Question 6: Calculate a “tacked” set of variables depending on the tack of the boat, so that sailors don’t need to think about what tack they’re on when looking at measurements. And show the results in a visualisation.


## Question 7: Given a set of tacks (in CSV), and train a model to explain the key features of these tacks when optimizing for vmg. Show appropriate visualisations to explain your conclusions.

## Question 8: Give insights on the racing on what made a team win or underperform in the race.